# Experiment: Sequential Anisotropic Damping Simulator Report

Objective:
- Test whether rotated finite-window anisotropic damping produces complementary residual branches without hard-coded collapse or Bell probabilities.
- Prioritize projectivity, residual purity, aligned support, and marginal stability before treating CHSH as a secondary summary.


In [ ]:
# Setup: locate artifacts, ensure plots exist, and load notebook dependencies
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "sim"
artifact_override = os.environ.get("SIM_ARTIFACT_DIR")
if artifact_override:
    ARTIFACT_DIR = Path(artifact_override).expanduser().resolve()
else:
    candidates = sorted(ARTIFACT_ROOT.glob("*"))
    if not candidates:
        raise FileNotFoundError(
            "No simulation artifacts found. Run scripts/run_sweeps.py first or set SIM_ARTIFACT_DIR."
        )
    ARTIFACT_DIR = candidates[-1]

summary_path = ARTIFACT_DIR / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing summary.json in {ARTIFACT_DIR}")

plots_dir = ARTIFACT_DIR / "plots"
if not plots_dir.exists():
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "analyze_results.py"), str(ARTIFACT_DIR)],
        check=True,
    )

ARTIFACT_DIR


## Plan

- Load the latest saved sweep artifacts instead of rerunning the simulator from the notebook.
- Surface the key single-analyzer and sequential summaries that drive the brief's success criteria.
- Display the generated plots and direct report answers from `summary.json` so the notebook remains a thin, reproducible report layer.


In [ ]:
# Load saved tables and show a compact run overview
def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

summary = json.loads(summary_path.read_text(encoding="utf-8"))
single_summary = load_csv(ARTIFACT_DIR / "single" / "summary.csv")
single_ratio_summary = load_csv(ARTIFACT_DIR / "single" / "ratio_summary.csv")
rule_summary = load_csv(ARTIFACT_DIR / "sequential" / "rule_summary.csv")
drift_summary = load_csv(ARTIFACT_DIR / "sequential" / "drift_summary.csv")
chsh_summary = load_csv(ARTIFACT_DIR / "sequential" / "chsh_summary.csv")

overview = pd.DataFrame(
    [
        {
            "artifact_dir": str(ARTIFACT_DIR),
            "modes": ", ".join(summary.get("modes", [])),
            "single_rows": len(single_summary),
            "sequential_rows": len(rule_summary),
            "plot_count": len(summary.get("plots", {})),
        }
    ]
)
display(overview)

best_single = (
    single_summary.sort_values("projectivity_score", ascending=False).head(5)
    if not single_summary.empty
    else pd.DataFrame()
)
best_chsh = (
    chsh_summary.sort_values("chsh", ascending=False).head(5)
    if not chsh_summary.empty
    else pd.DataFrame()
)
display(best_single)
display(best_chsh)


## Results

- The tables below answer the brief's direct report questions using the saved analysis summary.
- The follow-up tables expose the lowest-drift rules and the strongest CHSH rows so the interpretation stays tied to the actual artifact set.


In [ ]:
# Report answers, failure labels, and rule-level diagnostics
display(Markdown("## Final Report Questions"))
for key, answer in summary.get("report_answers", {}).items():
    display(Markdown(f"- **{key}**: {answer}"))

display(Markdown("## Failure Modes"))
failure_table = pd.DataFrame.from_dict(summary.get("failure_modes", {}), orient="index")
display(failure_table)

lowest_drift = (
    drift_summary.sort_values(["alice_drift_max", "bob_drift_max"]).head(10)
    if not drift_summary.empty
    else pd.DataFrame()
)
display(Markdown("## Lowest-Drift Sequential Rows"))
display(lowest_drift)


In [ ]:
# Display the saved plot set inline
plot_files = [
    ("Projectivity vs anisotropy", "projectivity-vs-anisotropy.png"),
    ("Residual quality vs anisotropy", "residual-quality-vs-anisotropy.png"),
    ("Single-analyzer branch response", "single-branch-response-vs-angle.png"),
    ("State clouds before/after", "state-clouds-before-after.png"),
    ("Marginal drift vs remote angle", "marginal-drift-vs-remote-angle.png"),
    ("Aligned same-sign mass", "aligned-same-sign-mass-vs-anisotropy.png"),
    ("Correlation vs delta", "correlation-vs-delta.png"),
    ("CHSH vs anisotropy", "chsh-vs-anisotropy.png"),
]

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
for ax, (title, filename) in zip(axes.flat, plot_files):
    plot_path = plots_dir / filename
    if plot_path.exists():
        ax.imshow(mpimg.imread(plot_path))
        ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f"Missing plot:\n{filename}", ha="center", va="center")
    ax.axis("off")
fig.tight_layout()
plt.show()


## Next steps

- Re-run `scripts/run_sweeps.py` with larger sample counts once a promising anisotropy/window regime appears.
- Use `SIM_ARTIFACT_DIR=/abs/path/to/artifact` when you want this notebook pinned to a specific run instead of the latest artifact directory.
- If the residual-branch score stays weak at strong anisotropy, stop tuning CHSH-facing readouts and revisit the damping mechanism itself.
